In [1]:
# Notebook 02: L2a Dummy Data Generation
# Produces L2a_daily.csv, L2a_daily.json, ../data/L2a_weekly_updated.csv, ../data/L2a_weekly_updated.json
 
# Cell 1: Imports and global constants
# All window, boundary, and date range settings defined here only

import pandas as pd
import numpy as np
import json
from scipy.stats import truncnorm

In [2]:
# Cell 2: Date spine and daily L1-level totals
# Totals used as day denominators when distributing SF across categories

SEED       = 77
RNG        = np.random.default_rng(SEED)
START      = "2023-01-01"
END        = "2023-12-31"
BIN_ORIGIN = pd.Timestamp("2023-01-01")  # Jan-01 anchor for all 7-day bins
N_BINS     = 52                           # 52 complete bins Jan-01 to Dec-29
ZERO_DATES = ["2023-12-25"]              # set to zero before binning
NULL_DATES = []                           # set to NaN before binning
METRICS    = ["GHGE", "LU", "WU"]

print("Constants loaded")
print(f"Bin origin: {BIN_ORIGIN.date()}, bins: {N_BINS}")
print(f"Output range: Jan-01 to Dec-29 ({N_BINS} weekly bins x {65} categories)")
 
dates        = pd.date_range(START, END, freq="D")
N            = len(dates)
day_idx      = np.arange(N)
 
def trunc(mean, std, low, high, size):
    a = (low  - mean) / std
    b = (high - mean) / std
    return truncnorm.rvs(a, b, loc=mean, scale=std, size=size, random_state=SEED)
 
seasonal     = 1.0 + 0.18 * np.sin((day_idx / N) * 2 * np.pi + np.pi * 0.3)
is_weekend   = pd.Series(dates).dt.dayofweek.isin([5, 6]).astype(float).values
weekend_bump = 1.0 + 0.08 * is_weekend
 
total_GHGE_SF = trunc(820_000,      95_000,     500_000,    1_100_000,    N) * seasonal * weekend_bump
total_LU_SF   = trunc(2_100_000,   480_000,     900_000,    3_400_000,    N) * seasonal * weekend_bump
total_WU_SF   = trunc(140_000_000, 28_000_000,  60_000_000, 200_000_000,  N) * seasonal * weekend_bump
 
print(f"Date spine: {N} days, {dates[0].date()} to {dates[-1].date()}")

Constants loaded
Bin origin: 2023-01-01, bins: 52
Output range: Jan-01 to Dec-29 (52 weekly bins x 65 categories)
Date spine: 365 days, 2023-01-01 to 2023-12-31


In [3]:
# Cell 3: LCFS category names - 65 categories, ONS LCFS classification
# Names match the original L2a dummy file exactly

LCFS_CATS = [
    "Apples", "Bacon and ham", "Baker's yeast, dessert preparations, soups",
    "Bananas", "Beef", "Berries", "Bread", "Buns, crispbread and biscuits",
    "Butter", "Cabbages", "Cakes and puddings", "Cheese and curd", "Chocolate",
    "Citrus fruits", "Cocoa and powdered chocolate", "Coffee",
    "Confectionery products", "Dairy alternative", "Dried fruit and nuts",
    "Dried vegetables", "Edible oils and other edible animal fats",
    "Edibles ices and ice cream", "Eggs", "Fish", "Fresh vegetables",
    "Fruit and vegetable juices", "Jams, marmalades", "Lamb",
    "Leaf and stem vegetables",
    "Margarine, other vegetable fats and peanut butter", "Meat alternative",
    "Milk", "Mineral or spring waters", "Offal, pate etc", "Olive oil",
    "Other breads and cereals", "Other fresh, chilled or frozen fruits",
    "Other fresh, chilled, or frozen edible meat",
    "Other milk products", "Other preserved or processed fish & seafood",
    "Other preserved or processed meat", "Other preserved or processed veg",
    "Other sugar products", "Pasta products", "Pastry (savoury)", "Pears",
    "Pork", "Potatoes", "Poultry", "Preserved fruits and fruit based products",
    "Preserved milk", "Ready meals", "Rice",
    "Root crops, non-starchy bulbs and mushrooms",
    "Salt, spices, herbs & other food products", "Sauces, condiments",
    "Sausages", "Savoury snacks", "Seafood, dried, smoked or salted fish",
    "Soft drinks (inc. ready to drink fruit drinks)", "Stone fruits", "Sugar",
    "Tea", "Vegetable grown for their fruit", "Yoghurt",
]
N_CATS = len(LCFS_CATS)
print(f"Categories: {N_CATS}  (expected 65)")

Categories: 65  (expected 65)


In [4]:
# Cell 4: Generate daily L2a rows for all categories and dates
# Dirichlet shares distribute each day total independently per metric

rows = []
for i, d in enumerate(dates):
    date_str    = d.strftime("%Y-%m-%d")
    ghge_shares = RNG.dirichlet(np.full(N_CATS, 1.6))
    lu_shares   = RNG.dirichlet(np.full(N_CATS, 1.4))
    wu_shares   = RNG.dirichlet(np.full(N_CATS, 1.8))
 
    cat_GHGE = ghge_shares * total_GHGE_SF[i]
    cat_LU   = lu_shares   * total_LU_SF[i]
    cat_WU   = wu_shares   * total_WU_SF[i]
 
    cat_GHGE_pkg = np.clip(trunc(5.2,   1.1,  2.0,  9.0,  N_CATS) + RNG.normal(0, 1.0,  N_CATS), 0.5,  25.0)
    cat_LU_pkg   = np.clip(trunc(11.8,  2.6,  4.0,  22.0, N_CATS) + RNG.normal(0, 2.0,  N_CATS), 1.0,  50.0)
    cat_WU_pkg   = np.clip(trunc(270.0, 55.0, 80.0, 500.0, N_CATS) + RNG.normal(0, 35.0, N_CATS), 10.0, 800.0)
 
    ghge_ranks = pd.Series(cat_GHGE).rank(ascending=False, method="min").astype(int).values
    lu_ranks   = pd.Series(cat_LU).rank(ascending=False, method="min").astype(int).values
    wu_ranks   = pd.Series(cat_WU).rank(ascending=False, method="min").astype(int).values
 
    for j, cat in enumerate(LCFS_CATS):
        rows.append({
            "date"              : date_str,
            "lcfs_cat"          : cat,
            "total_GHGE_SF"     : round(float(cat_GHGE[j]),    4),
            "total_LU_SF"       : round(float(cat_LU[j]),      4),
            "total_WU_SF"       : round(float(cat_WU[j]),      4),
            "avg_GHGE_perkg"    : round(float(cat_GHGE_pkg[j]),6),
            "avg_LU_perkg"      : round(float(cat_LU_pkg[j]),  6),
            "avg_WU_perkg"      : round(float(cat_WU_pkg[j]),  6),
            "GHGE_rank_on_day"  : int(ghge_ranks[j]),
            "LU_rank_on_day"    : int(lu_ranks[j]),
            "WU_rank_on_day"    : int(wu_ranks[j]),
        })
 
L2a = pd.DataFrame(rows)
L2a = L2a.sort_values(["lcfs_cat", "date"]).reset_index(drop=True)
print(f"L2a shape: {L2a.shape}  (expected {N * N_CATS} rows)") 

L2a shape: (23725, 11)  (expected 23725 rows)


In [5]:
# Cell 5: Apply closure masks and sort
# Zero dates included as 0 in rolling windows; null dates excluded entirely

SF_COLS      = ["total_GHGE_SF", "total_LU_SF", "total_WU_SF"]
PKG_COLS     = ["avg_GHGE_perkg", "avg_LU_perkg", "avg_WU_perkg"]
ALL_VAL_COLS = SF_COLS + PKG_COLS
 
for d in ZERO_DATES:
    mask = L2a["date"] == d
    L2a.loc[mask, ALL_VAL_COLS] = 0.0
    print(f"Zero mask: {d} ({int(mask.sum())} rows)")
 
for d in NULL_DATES:
    mask = L2a["date"] == d
    L2a.loc[mask, ALL_VAL_COLS] = np.nan
    print(f"Null mask: {d} ({int(mask.sum())} rows)")

Zero mask: 2023-12-25 (65 rows)


In [6]:
# ==============================================================================
# Cell 6: Aggregate daily values into 52 Jan-01-anchored 7-day bins per category
# Resample per category then concatenate -- ensures consistent bin labels.
# BIN_ORIGIN = Jan-01-2023: bin 1 = Jan-01 to Jan-07, bin 2 = Jan-08 to Jan-14.
# Closure day zeros (Dec-25) contribute to their bin mean -- expected behaviour.
# N_BINS = 52: excludes the partial Dec-30 to Dec-31 trailing bin.
# Output: N_BINS x N_CATS = 3,380 rows.
# ==============================================================================

# Source columns and their output names after binning.
SF_COLS  = ["total_GHGE_SF", "total_LU_SF", "total_WU_SF"]
PKG_COLS = ["avg_GHGE_perkg", "avg_LU_perkg", "avg_WU_perkg"]

BIN_RENAME = {
    "total_GHGE_SF"   : "GHGE_weekly_mean",
    "total_LU_SF"     : "LU_weekly_mean",
    "total_WU_SF"     : "WU_weekly_mean",
    "avg_GHGE_perkg"  : "GHGE_perkg_weekly_mean",
    "avg_LU_perkg"    : "LU_perkg_weekly_mean",
    "avg_WU_perkg"    : "WU_perkg_weekly_mean",
}

# Resample each category independently to Jan-01-anchored 7-day bins.
# Set date as datetime index for resample, then loop per category.
L2a["date_dt"] = pd.to_datetime(L2a["date"])
L2a_dt         = L2a.set_index("date_dt")

bin_frames = []
for cat, cat_df in L2a_dt.groupby("lcfs_cat"):
    binned = (
        cat_df[SF_COLS + PKG_COLS]
        .resample("7D", origin=BIN_ORIGIN)
        .mean()
        .reset_index()
        .rename(columns={"date_dt": "week_start"})
    )
    binned["lcfs_cat"] = cat
    bin_frames.append(binned)

weekly = pd.concat(bin_frames, ignore_index=True)

# Keep N_BINS complete bins -- discards the partial Dec-30 to Dec-31 bin.
valid_weeks = sorted(weekly["week_start"].unique())[:N_BINS]
weekly      = weekly[weekly["week_start"].isin(valid_weeks)].copy()
weekly      = weekly.sort_values(["week_start", "lcfs_cat"]).reset_index(drop=True)

# Convert week_start to string and rename value columns to output names.
weekly["week_start"] = weekly["week_start"].dt.strftime("%Y-%m-%d")
weekly = weekly.rename(columns=BIN_RENAME)

# Round all value columns.
for col in BIN_RENAME.values():
    weekly[col] = weekly[col].round(6)

# Calendar month: first day of each bin determines the month (0-indexed).
weekly["calendar_month"] = (
    pd.to_datetime(weekly["week_start"]).dt.month - 1
)

print(f"Weekly shape: {weekly.shape}  (expected {N_BINS * N_CATS} rows)")
print(f"First bin: {weekly['week_start'].iloc[0]}")
print(f"Last bin:  {weekly['week_start'].iloc[-1]}")
print(f"Unique bins: {weekly['week_start'].nunique()}")
print(f"Unique cats: {weekly['lcfs_cat'].nunique()}")

Weekly shape: (3380, 9)  (expected 3380 rows)
First bin: 2023-01-01
Last bin:  2023-12-24
Unique bins: 52
Unique cats: 65


In [7]:
# ==============================================================================
# Cell 7: Annual statistics per category from weekly bin values
# All reference values computed from the 52-bin weekly data.
# Attached to the weekly DataFrame so every row carries its category's
# reference values. D3 reads them directly -- no rollup needed in browser.
# ==============================================================================

for m in METRICS:
    sf_col  = f"{m}_weekly_mean"
    pkg_col = f"{m}_perkg_weekly_mean"

    # Annual mean: mean of the 52 bin means per category.
    cat_sf_mean  = weekly.groupby("lcfs_cat")[sf_col].transform("mean")
    cat_pkg_mean = weekly.groupby("lcfs_cat")[pkg_col].transform("mean")
    cat_pkg_min  = weekly.groupby("lcfs_cat")[pkg_col].transform("min")
    cat_pkg_max  = weekly.groupby("lcfs_cat")[pkg_col].transform("max")

    weekly[f"{m}_annual_mean"]    = cat_sf_mean.round(6)
    weekly[f"{m}_annual_min"]     = weekly.groupby("lcfs_cat")[sf_col].transform("min").round(6)
    weekly[f"{m}_annual_max"]     = weekly.groupby("lcfs_cat")[sf_col].transform("max").round(6)

    # Annual share: category bin-mean total as fraction of grand total across
    # all 65 categories across all 52 bins.
    grand_total               = weekly[sf_col].sum()
    cat_total                 = weekly.groupby("lcfs_cat")[sf_col].transform("sum")
    weekly[f"{m}_annual_share"] = ((cat_total / grand_total) * 100).round(6)

    # Annual rank by bin-mean annual mean: 1 = highest impact category.
    sf_means                  = weekly.groupby("lcfs_cat")[sf_col].mean()
    weekly[f"{m}_annual_rank"] = weekly["lcfs_cat"].map(
        sf_means.rank(ascending=False, method="min").astype(int)
    )

    # Pct change: each bin's value vs that category's annual mean.
    weekly[f"{m}_pct_from_annual_mean"] = (
        ((weekly[sf_col] - cat_sf_mean) / cat_sf_mean) * 100
    ).round(6)

    # Per-kg annual stats from bin means.
    weekly[f"{m}_perkg_annual_mean"] = cat_pkg_mean.round(6)
    weekly[f"{m}_perkg_annual_min"]  = cat_pkg_min.round(6)
    weekly[f"{m}_perkg_annual_max"]  = cat_pkg_max.round(6)

    # Max absolute deviation: diverging colour scale half-width.
    weekly[f"{m}_perkg_max_abs_dev"] = np.maximum(
        (cat_pkg_max - cat_pkg_mean).abs(),
        (cat_pkg_mean - cat_pkg_min).abs()
    ).round(6)

    pkg_means                          = weekly.groupby("lcfs_cat")[pkg_col].mean()
    weekly[f"{m}_perkg_annual_rank"]   = weekly["lcfs_cat"].map(
        pkg_means.rank(ascending=False, method="min").astype(int)
    )

    weekly[f"{m}_perkg_pct_from_annual_mean"] = (
        ((weekly[pkg_col] - cat_pkg_mean) / cat_pkg_mean) * 100
    ).round(6)

# Weekly ranks: rank each category within each bin by weekly_mean.
for m in METRICS:
    weekly[f"{m}_weekly_rank"] = (
        weekly.groupby("week_start")[f"{m}_weekly_mean"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

print(f"Annual stats complete -- weekly columns: {weekly.shape[1]}")
print(f"Weekly shape: {weekly.shape}")

Annual stats complete -- weekly columns: 48
Weekly shape: (3380, 48)


In [8]:
# # Cell 8: Weekly aggregation from smoothed values
# # Filtered to DATA_START-DATA_END first so all roll7 inputs are non-null

# def get_week_start(date_str):
#     # Jan-01 anchored 7-day bins regardless of day of week.
#     # Week 1: Jan-01 to Jan-07, Week 2: Jan-08 to Jan-14, etc.
#     d    = pd.Timestamp(date_str)
#     jan1 = pd.Timestamp(d.year, 1, 1)
#     bin_num    = (d - jan1).days // 7
#     week_start = jan1 + pd.Timedelta(days=7 * bin_num)
#     return week_start.strftime("%Y-%m-%d")

# # Assign Jan-01-anchored week_start for all L2a rows
# L2a["week_start"] = L2a["date"].apply(get_week_start)
 
# # Majority calendar month per week (0-indexed: Jan=0, Dec=11)
# week_month_map = (
#     L2a.groupby("week_start")["date"]
#     .apply(lambda s: (pd.to_datetime(s).dt.month - 1).value_counts().idxmax())
#     .reset_index()
#     .rename(columns={"date": "calendar_month"})
# )
 
# # Filter to valid smoothed range before groupby - ensures no NaN weekly means
# L2a_smooth = L2a[
#     (L2a["date"] >= DATA_START) &
#     (L2a["date"] <= DATA_END)
# ].copy()
 
# grp      = L2a_smooth.groupby(["week_start", "lcfs_cat"])
# agg_dict = {}
# for m in METRICS:
#     agg_dict[f"{m}_weekly_mean"]       = (f"roll7_{m}_SF",    "mean")
#     agg_dict[f"{m}_perkg_weekly_mean"] = (f"roll7_{m}_perkg", "mean")
 
# weekly = grp.agg(**{k: v for k, v in agg_dict.items()}).reset_index()
# weekly = weekly.sort_values(["week_start", "lcfs_cat"]).reset_index(drop=True)
 
# # Weekly ranks computed only on filtered weeks - no NaN means so astype(int) is safe
# for m in METRICS:
#     weekly[f"{m}_weekly_rank"] = (
#         weekly.groupby("week_start")[f"{m}_weekly_mean"]
#         .rank(ascending=False, method="min")
#         .astype(int)
#     )
 
# # Merge calendar month and all annual reference columns
# annual_ref_cols = (
#     ["lcfs_cat"]
#     + [f"{m}_{s}" for m in METRICS for s in [
#         "annual_mean", "annual_min", "annual_max", "annual_share", "annual_rank",
#         "perkg_annual_mean", "perkg_annual_min", "perkg_annual_max",
#         "perkg_max_abs_dev", "perkg_annual_rank",
#     ]]
# )
# annual_ref = (
#     L2a[annual_ref_cols]
#     .drop_duplicates("lcfs_cat")
#     .reset_index(drop=True)
# )
 
# weekly = weekly.merge(annual_ref,       on="lcfs_cat",   how="left")
# weekly = weekly.merge(week_month_map,   on="week_start", how="left")
# weekly["calendar_month"] = weekly["calendar_month"].astype(int)
 
# for m in METRICS:
#     weekly[f"{m}_pct_from_annual_mean"] = (
#         ((weekly[f"{m}_weekly_mean"] - weekly[f"{m}_annual_mean"])
#          / weekly[f"{m}_annual_mean"]) * 100
#     ).round(6)
#     weekly[f"{m}_perkg_pct_from_annual_mean"] = (
#         ((weekly[f"{m}_perkg_weekly_mean"] - weekly[f"{m}_perkg_annual_mean"])
#          / weekly[f"{m}_perkg_annual_mean"]) * 100
#     ).round(6)
 
# for col in [c for c in weekly.columns if "mean" in c.lower()]:
#     weekly[col] = weekly[col].round(6)
 
# print(f"Weekly shape: {weekly.shape}")
# print(f"Weeks: {weekly['week_start'].nunique()}, Categories: {weekly['lcfs_cat'].nunique()}")

# ==============================================================================
# Cell 8: Weekly file is already complete from Cells 6 and 7
# No additional aggregation needed -- bins, stats, and ranks all attached.
# ==============================================================================

print(f"Weekly ready: {len(weekly)} rows, {weekly['week_start'].nunique()} bins, "
      f"{weekly['lcfs_cat'].nunique()} categories")

Weekly ready: 3380 rows, 52 bins, 65 categories


In [9]:
# ==============================================================================
# Cell 9: Define weekly export column list
# Daily file removed entirely -- heatmap chart uses weekly only.
# All 52 bins already complete from Cell 6. No date filter needed.
# ==============================================================================

WEEKLY_COLS = (
    ["week_start", "lcfs_cat", "calendar_month"]
    + [f"{m}_weekly_mean"       for m in METRICS]
    + [f"{m}_perkg_weekly_mean" for m in METRICS]
    + [f"{m}_weekly_rank"       for m in METRICS]
    + [f"{m}_{s}" for m in METRICS for s in [
        "annual_mean", "annual_min", "annual_max", "annual_share", "annual_rank",
        "pct_from_annual_mean",
        "perkg_annual_mean", "perkg_annual_min", "perkg_annual_max",
        "perkg_max_abs_dev", "perkg_annual_rank", "perkg_pct_from_annual_mean",
    ]]
)

L2a_weekly = weekly[WEEKLY_COLS].copy()

missing = [c for c in WEEKLY_COLS if c not in weekly.columns]
print(f"Missing columns: {missing if missing else 'None'}")
print(f"Weekly rows: {len(L2a_weekly)} (expected {N_BINS * N_CATS})")
print(f"Bins: {L2a_weekly['week_start'].nunique()}, Cats: {L2a_weekly['lcfs_cat'].nunique()}")

Missing columns: None
Weekly rows: 3380 (expected 3380)
Bins: 52, Cats: 65


In [10]:
# ==============================================================================
# Cell 10: Export weekly file as CSV and JSON with verification
# Daily file removed -- heatmap uses weekly only.
# ==============================================================================

L2a_weekly.to_csv("../data/L2a_weekly_updated.csv", index=False)
print("Saved: ../data/L2a_weekly_updated.csv")

L2a_weekly.to_json("../data/L2a_weekly_updated.json", orient="records", indent=2)
print("Saved: ../data/L2a_weekly_updated.json")

with open("../data/L2a_weekly_updated.json") as f:
    w_check = json.load(f)

w_weeks = sorted(set(r["week_start"] for r in w_check))
print(f"Records:     {len(w_check)} (expected {N_BINS * N_CATS})")
print(f"Week range:  {w_weeks[0]} -> {w_weeks[-1]}")
print(f"Unique bins: {len(w_weeks)}")

# Null check on all weekly mean columns
print("Null check on weekly mean columns (all must be 0):")
mean_cols = (
    [f"{m}_weekly_mean"       for m in METRICS] +
    [f"{m}_perkg_weekly_mean" for m in METRICS]
)
for col in mean_cols:
    nulls = sum(1 for r in w_check if r.get(col) is None)
    print(f"  {col:<30} {'OK' if nulls == 0 else 'NULLS: ' + str(nulls)}")

# Sample row from bin 1
sample = next(r for r in w_check if r["week_start"] == w_weeks[0]
              and r["lcfs_cat"] == "Beef")
print(f"\nSample row (bin 1, Beef):")
for col in ["week_start", "GHGE_weekly_mean", "GHGE_perkg_weekly_mean",
            "GHGE_annual_mean", "GHGE_annual_share",
            "GHGE_pct_from_annual_mean", "GHGE_weekly_rank",
            "calendar_month"]:
    print(f"  {col:<35} {sample.get(col)}")

print("\nAll exports complete")

Saved: ../data/L2a_weekly_updated.csv
Saved: ../data/L2a_weekly_updated.json
Records:     3380 (expected 3380)
Week range:  2023-01-01 -> 2023-12-24
Unique bins: 52
Null check on weekly mean columns (all must be 0):
  GHGE_weekly_mean               OK
  LU_weekly_mean                 OK
  WU_weekly_mean                 OK
  GHGE_perkg_weekly_mean         OK
  LU_perkg_weekly_mean           OK
  WU_perkg_weekly_mean           OK

Sample row (bin 1, Beef):
  week_start                          2023-01-01
  GHGE_weekly_mean                    20161.408729
  GHGE_perkg_weekly_mean              3.976179
  GHGE_annual_mean                    13383.713145
  GHGE_annual_share                   1.611931
  GHGE_pct_from_annual_mean           50.641369
  GHGE_weekly_rank                    8
  calendar_month                      0

All exports complete


In [11]:
# P4 spot check: annual share must sum to 100 across all 65 cats for each bin.
# Use one representative bin. If the sum is not 100 the grand_total denominator
# is wrong in Cell 7.

for m in METRICS:
    share_col = f"{m}_annual_share"
    # Annual share is constant across bins for the same category.
    # Sum across all 65 categories for bin 1.
    bin1_share_sum = (
        L2a_weekly[L2a_weekly["week_start"] == w_weeks[0]][share_col].sum()
    )
    flag = "OK" if abs(bin1_share_sum - 100.0) < 0.01 else "MISMATCH"
    print(f"{m} annual share sum (bin 1): {bin1_share_sum:.4f}  {flag}")

GHGE annual share sum (bin 1): 100.0000  OK
LU annual share sum (bin 1): 100.0000  OK
WU annual share sum (bin 1): 100.0000  OK


In [12]:
# P4 spot check: pct_from_annual_mean should average near zero per category
# across all 52 bins (by definition -- deviation from the category's own mean).
# Check Beef as representative category.

for m in METRICS:
    pct_col  = f"{m}_pct_from_annual_mean"
    beef_pct = L2a_weekly[L2a_weekly["lcfs_cat"] == "Beef"][pct_col].mean()
    flag     = "OK" if abs(beef_pct) < 0.5 else "MISMATCH -- check annual mean calc"
    print(f"{m} Beef pct_from_mean average across 52 bins: {beef_pct:.4f}  {flag}")

GHGE Beef pct_from_mean average across 52 bins: 0.0000  OK
LU Beef pct_from_mean average across 52 bins: 0.0000  OK
WU Beef pct_from_mean average across 52 bins: 0.0000  OK


In [14]:
# P4-4 revised: verify the zero mask is applied correctly and
# that Dec-25 is contributing zero to the bin 52 mean.
# Do not compare bin 52 to bin 51 -- surrounding days may be
# high enough that the overall bin mean is above bin 51 regardless.

# Step 1: confirm Dec-25 is 0 in the raw L2a daily data for Beef.
beef_dec25 = L2a[
    (L2a["date"] == "2023-12-25") &
    (L2a["lcfs_cat"] == "Beef")
]["total_GHGE_SF"].values

if len(beef_dec25) == 0:
    print("Dec-25 Beef row not found in L2a -- check ZERO_DATES mask ran before Cell 6")
elif beef_dec25[0] == 0.0:
    print(f"Dec-25 Beef total_GHGE_SF: {beef_dec25[0]}  OK -- zero mask applied correctly")
else:
    print(f"Dec-25 Beef total_GHGE_SF: {beef_dec25[0]}  MISMATCH -- zero mask did not apply")

# Step 2: confirm bin 52 mean is lower than the 6-day mean excluding Dec-25.
# If zero mask is applied the 7-day mean must be (6/7) * 6-day mean.
beef_bin52_days = L2a[
    (L2a["date"] >= "2023-12-24") &
    (L2a["date"] <= "2023-12-30") &
    (L2a["lcfs_cat"] == "Beef")
][["date", "total_GHGE_SF"]]

mean_7day = beef_bin52_days["total_GHGE_SF"].mean()
mean_6day = beef_bin52_days[beef_bin52_days["date"] != "2023-12-25"]["total_GHGE_SF"].mean()
exported  = L2a_weekly[
    (L2a_weekly["week_start"] == "2023-12-24") &
    (L2a_weekly["lcfs_cat"] == "Beef")
]["GHGE_weekly_mean"].values[0]

print(f"\nBin 52 (Dec-24 to Dec-30) Beef GHGE:")
print(f"  6-day mean (excl Dec-25):  {mean_6day:.2f}")
print(f"  7-day mean (incl zero):    {mean_7day:.2f}")
print(f"  Exported weekly mean:      {exported:.2f}")
print(f"  7-day < 6-day: {'YES -- zero reduces mean correctly' if mean_7day < mean_6day else 'NO -- check mask'}")
print(f"  Exported matches 7-day:   {'YES' if abs(exported - mean_7day) < 0.01 else 'NO -- check bin aggregation'}")

Dec-25 Beef total_GHGE_SF: 0.0  OK -- zero mask applied correctly

Bin 52 (Dec-24 to Dec-30) Beef GHGE:
  6-day mean (excl Dec-25):  19038.49
  7-day mean (incl zero):    16318.71
  Exported weekly mean:      16318.71
  7-day < 6-day: YES -- zero reduces mean correctly
  Exported matches 7-day:   YES
